# MobileNet — Efficient CNNs for Mobile Applications (TensorFlow / Keras)
**Paper:** MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications (Howard et al., 2017)

| Property | Value |
|----------|-------|
| Framework | TensorFlow / Keras |
| Block type | Depthwise Separable Conv (DWConv + PWConv) |
| DW Blocks | 13 |
| Parameters | ~4.2M |
| Top-1 (ImageNet) | ~70.6% |
| Input size | 224×224 |
| Preprocessing | mobilenet.preprocess_input → x/127.5 − 1.0 → [−1, 1] |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {bool(tf.config.list_physical_devices("GPU"))}')

In [ ]:
# Cell 2 — MobileNet Architecture (from scratch)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def _dw_block(x, filters, stride, block_id, alpha=1.0):
    """Depthwise Separable Convolution block.
    DWConv3x3(stride) + BN + ReLU6  then  PWConv1x1 + BN + ReLU6.
    alpha: width multiplier (scales all filter counts).
    """
    filters = int(filters * alpha)
    x = layers.DepthwiseConv2D(
        3, strides=stride, padding='same', use_bias=False,
        name=f'conv_dw_{block_id}')(x)
    x = layers.BatchNormalization(name=f'conv_dw_{block_id}_bn')(x)
    x = layers.ReLU(6., name=f'conv_dw_{block_id}_relu')(x)

    x = layers.Conv2D(
        filters, 1, padding='same', use_bias=False,
        name=f'conv_pw_{block_id}')(x)
    x = layers.BatchNormalization(name=f'conv_pw_{block_id}_bn')(x)
    x = layers.ReLU(6., name=f'conv_pw_{block_id}_relu')(x)
    return x


def build_mobilenet(num_classes=1000, input_shape=(224, 224, 3), alpha=1.0):
    """
    MobileNet — Efficient Convolutional Neural Networks for Mobile Applications.
    Paper: MobileNets: Efficient Convolutional Neural Networks for Mobile Vision
           Applications (Howard et al., 2017).

    Replaces standard convolutions with Depthwise Separable Convolutions:
      Standard Conv(k,k,c_in,c_out) -> DWConv(k,k,c_in) + PWConv(1,1,c_in,c_out)
      Parameter reduction: ~8-9x  |  FLOP reduction: ~8-9x

    alpha: width multiplier (scales all filter counts, default=1.0)
    Resolution multiplier rho: controlled via input_shape.

    Architecture:
      Stem       : Conv3x3/2  (32 ch)
      DW Block 1 : DWSep (64,  stride=1)
      DW Block 2 : DWSep (128, stride=2)
      DW Block 3 : DWSep (128, stride=1)
      DW Block 4 : DWSep (256, stride=2)
      DW Block 5 : DWSep (256, stride=1)
      DW Block 6 : DWSep (512, stride=2)
      DW Blocks 7-11: DWSep (512, stride=1) x5
      DW Block 12: DWSep (1024, stride=2)
      DW Block 13: DWSep (1024, stride=1)
      GlobalAvgPool -> Dense(num_classes, softmax)
    """
    stem_filters = int(32 * alpha)

    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(stem_filters, 3, strides=2, padding='same',
                      use_bias=False, name='conv1')(inputs)
    x = layers.BatchNormalization(name='conv1_bn')(x)
    x = layers.ReLU(6., name='conv1_relu')(x)

    # 13 Depthwise Separable blocks
    x = _dw_block(x,  64, stride=1, block_id=1,  alpha=alpha)
    x = _dw_block(x, 128, stride=2, block_id=2,  alpha=alpha)
    x = _dw_block(x, 128, stride=1, block_id=3,  alpha=alpha)
    x = _dw_block(x, 256, stride=2, block_id=4,  alpha=alpha)
    x = _dw_block(x, 256, stride=1, block_id=5,  alpha=alpha)
    x = _dw_block(x, 512, stride=2, block_id=6,  alpha=alpha)
    x = _dw_block(x, 512, stride=1, block_id=7,  alpha=alpha)
    x = _dw_block(x, 512, stride=1, block_id=8,  alpha=alpha)
    x = _dw_block(x, 512, stride=1, block_id=9,  alpha=alpha)
    x = _dw_block(x, 512, stride=1, block_id=10, alpha=alpha)
    x = _dw_block(x, 512, stride=1, block_id=11, alpha=alpha)
    x = _dw_block(x, 1024, stride=2, block_id=12, alpha=alpha)
    x = _dw_block(x, 1024, stride=1, block_id=13, alpha=alpha)

    x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
    outputs = layers.Dense(num_classes, activation='softmax',
                           name='predictions')(x)

    return keras.Model(inputs=inputs, outputs=outputs, name='mobilenet')


In [ ]:
NUM_CLASSES = 10
INPUT_SHAPE = (224, 224, 3)

model = build_mobilenet(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE)
model.summary(line_length=120)

In [ ]:
dummy  = tf.random.normal((2, 224, 224, 3))
output = model(dummy, training=False)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
total_params     = model.count_params()
trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
non_trainable    = total_params - trainable_params
print(f"{'='*48}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"  Non-trainable         : {non_trainable:,}")
print(f"{'='*48}")

In [ ]:
print(f"{'Layer':<55} {'Output Shape':<30} {'Params':>10}")
print('-' * 98)
total = 0
for layer in model.layers:
    params = layer.count_params()
    total += params
    try:
        out_shape = str(layer.output_shape)
    except Exception:
        out_shape = 'multiple'
    print(f"{layer.name:<55} {out_shape:<30} {params:>10,}")
print('-' * 98)
print(f"{'TOTAL':<86} {total:>10,}")

---
## Training

In [ ]:
DATA_DIR   = './data'
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f'{DATA_DIR}/train', target_size=(224, 224),
    batch_size=BATCH_SIZE, class_mode='categorical',
)
val_gen = val_datagen.flow_from_directory(
    f'{DATA_DIR}/val', target_size=(224, 224),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False,
)
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {train_gen.samples}')
print(f'Val size   : {val_gen.samples}')

In [ ]:
EPOCHS = 30

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
callbacks = [
    keras.callbacks.ModelCheckpoint(
        'mobilenet_best.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.1, patience=5,
        min_lr=1e-7, verbose=1,
    ),
]
history = model.fit(
    train_gen, epochs=EPOCHS, validation_data=val_gen,
    callbacks=callbacks, verbose=1,
)
print('Best model saved: mobilenet_best.keras')

---
## Training Curves

In [ ]:
epochs_range = range(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MobileNet — Training Curves', fontsize=14, fontweight='bold')
axes[0].plot(epochs_range, history.history['loss'],         'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history.history['val_loss'],     'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_range, history.history['accuracy'],     'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Inference

In [ ]:
def predict_image(image_path, top_k=5):
    img    = keras.utils.load_img(image_path, target_size=(224, 224))
    arr    = keras.utils.img_to_array(img) / 255.0
    tensor = tf.expand_dims(arr, 0)
    probs  = model(tensor, training=False)[0].numpy()
    top_idx = probs.argsort()[::-1][:top_k]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input')
    axes[1].barh([CLASS_NAMES[i] for i in top_idx][::-1],
                 [probs[i]*100 for i in top_idx][::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    plt.tight_layout(); plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({probs[top_idx[0]]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
val_gen.reset()
y_pred     = model.predict(val_gen, steps=len(val_gen), verbose=1)
y_true     = val_gen.classes
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
print(f'Predictions shape : {y_pred.shape}')
print(f'Labels shape      : {y_true_bin.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred[:, i])
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {auc(fpr,tpr):.3f})')
macro_auc = roc_auc_score(y_true_bin, y_pred, average='macro', multi_class='ovr')
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'MobileNet — ROC AUC  (Macro AUC = {macro_auc:.3f})', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Macro AUC : {macro_auc:.4f}')